# Spectral Extraction

Three chapters of preparation come together here. Chapter 6 straightened the frame, Chapter 4
found where the source falls on it, and Chapter 7 removed the sky from underneath it. What
remains is a 2D frame in which the only significant signal is the object itself, spread over a
handful of rows around a known, curved path. **Extraction** is the step that collapses those rows
into a single number per dispersion column, a 1D spectrum.

The simplest extraction sums a few rows either side of the trace. That is `BoxcarExtract`: fast,
nearly free of assumptions, and close to optimal for a bright source. It has two weaknesses. It
weights a pixel in the faint wings of the profile as heavily as one at the peak, although the wing
pixel carries far more noise per unit signal, and it captures a fraction of the source's light
that depends on the profile width, which varies with wavelength. `HorneExtract` addresses both by
weighting each pixel by the signal it should contain and by its noise.

In [ ]:
import sys

sys.path.append("../src")

import numpy as np
import matplotlib.pyplot as plt
from astropy.modeling import models
from specreduce.background import Background
from specreduce.extract import BoxcarExtract, HorneExtract
from specreduce.tilt_solution import TiltSolution
from specreduce.tracing import FitTrace

from common import TILT_SOLUTION_FILE, fit_tilt_solution, read_data

plt.rc("figure", figsize=(8, 2.4), dpi=110)
plt.rc("font", size=9)

arcs, lamps, obj = read_data()
ny, nx = obj.data.shape
rows = np.arange(ny)

The three earlier steps run in one cell, with the settings their chapters chose: rectify with the
stored tilt solution, trace, and subtract the sky. The result carries its uncertainty end to end,
because `TiltSolution.resample` and `Background.sub_image()` both propagate it, and `HorneExtract`
needs it.

In [ ]:
if not TILT_SOLUTION_FILE.exists():  # normally written by Chapter 6
    fit_tilt_solution(arcs).to_asdf(TILT_SOLUTION_FILE)

solution = TiltSolution.from_asdf(TILT_SOLUTION_FILE)

rectified = solution.resample(obj)

trace = FitTrace(
    rectified, bins=32, peak_method="gaussian", trace_model=models.Chebyshev1D(4)
)
background = Background.two_sided(rectified, trace, separation=60, width=20)
image = background.sub_image()

trace_y = np.asarray(trace.trace)
print(
    f"image {image.flux.shape}, {image.flux.unit}, "
    f"uncertainty: {type(image.uncertainty).__name__}"
)

## Boxcar extraction

`BoxcarExtract` sums `width` rows centered on the trace, so the aperture follows the source as it
curves across the frame. Like the other extraction classes, it is configured when it is created
and runs when its `spectrum` property is read.

In [ ]:
boxcar = BoxcarExtract(image, trace, width=11)
spectrum = boxcar.spectrum

flux = spectrum.flux.value
error = spectrum.uncertainty.array

print(f"spectrum: {type(spectrum).__name__}, {flux.shape}, {spectrum.flux.unit}")
print(f"flux at column 512: {flux[512]:.0f} +/- {error[512]:.0f} e-")

The result is a `specutils` `Spectrum` with an uncertainty, on an axis that is still pixels:
extraction does not calibrate wavelength.

In [ ]:
fig, axs = plt.subplots(
    2, figsize=(8, 3.6), sharex="all", constrained_layout=True, height_ratios=[2, 1]
)

axs[0].plot(flux, lw=0.7)
axs[0].set_ylabel("Flux [e$^-$]")
axs[1].plot(flux / error, lw=0.7, c="C1")
axs[1].set(xlim=(0, nx - 1), xlabel="Dispersion [pix]", ylabel="SNR");


This is the eleven-row cut of Chapter 1, now with the sky removed and an uncertainty on every
point. The broad shape is the combined response of star, grism and detector, the dips are stellar
and telluric absorption, and the signal-to-noise falls toward both ends, where the grism
throughput drops.

### How wide should the aperture be?

Widening the aperture collects more of the source's light, but each added row brings its own read
noise and sky residual for progressively less signal, so the signal-to-noise has an optimum.

In [ ]:
widths = np.array([3, 5, 7, 9, 11, 13, 17, 21, 27, 35])
captured = np.zeros(widths.size)
snr = np.zeros(widths.size)

for i, width in enumerate(widths):
    s = BoxcarExtract(image, trace, width=width).spectrum
    f = s.flux.value
    e = s.uncertainty.array
    captured[i] = np.median(f)
    snr[i] = np.median(f / e)

reference = captured[-1]
print(f"{'width':>6}  {'median flux':>12}  {'% of widest':>12}  {'median SNR':>11}")
for width, f, s in zip(widths, captured, snr):
    print(f"{width:>6}  {f:>12.0f}  {100 * f / reference:>12.1f}  {s:>11.1f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 2.8), constrained_layout=True)
ax.plot(widths, snr, "o-", lw=0.9, ms=4)
ax.axvline(widths[snr.argmax()], c="0.6", lw=0.8, ls="--")
ax.text(
    widths[snr.argmax()] + 0.7,
    snr.min(),
    f"optimum, width {widths[snr.argmax()]}",
    size=8,
    c="0.4",
)
ax.set(xlabel="Aperture width [pix]", ylabel="Median SNR");

The signal-to-noise rises steeply to width 11 and then flattens: everything from 11 to 35 is
within 1 percent of the maximum, which falls at width 17. For a Gaussian profile a summed aperture
is optimal near 1.4 FWHM, about 5 pixels for this source's 3-pixel FWHM, when sky and read noise
dominate. The plateau here starts much wider because this bright source is photon-limited, so
widening the aperture keeps adding signal faster than noise, and because its profile has more
extended wings than a Gaussian. We adopt width 11, the narrowest
aperture on the plateau. It collects about 95 percent of what a width-35 aperture does, and the
wing pixels beyond it add signal and noise in nearly equal measure.

### Aperture losses depend on wavelength

Losing 5 percent of the light would be harmless if it were the same 5 percent everywhere. It is
not: seeing, focus and atmospheric dispersion vary with wavelength, so the profile width changes
across the detector and a fixed aperture captures a varying fraction. To measure it, we resample
every column onto a grid of offsets from the trace, which straightens the source, and compare the
flux inside each aperture with the total within 30 rows of the trace, in blocks of 128 columns.

In [ ]:
offsets = np.linspace(-30, 30, 241)
stack = np.array(
    [np.interp(trace_y[i] + offsets, rows, image.flux.value[:, i]) for i in range(nx)]
).T

block = 128
block_starts = range(0, nx, block)
total = np.array([stack[:, x : x + block].sum() for x in block_starts])

print(f"{'columns':>12}" + "".join(f"{f'width {w}':>11}" for w in (5, 9, 11, 15)))
for x, t in zip(block_starts, total):
    line = f"{f'{x}-{x + block}':>12}"
    for width in (5, 9, 11, 15):
        inside = np.abs(offsets) <= width / 2
        line += f"{stack[inside, x : x + block].sum() / t:>11.4f}"
    print(line)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 2.8), constrained_layout=True)

for width in (5, 9, 11, 15):
    inside = np.abs(offsets) <= width / 2
    fraction = (
        np.array([stack[inside, x : x + block].sum() for x in block_starts]) / total
    )
    ax.plot(
        np.array(block_starts) + block / 2,
        fraction,
        "o-",
        lw=0.9,
        ms=3,
        label=f"width {width}",
    )
    print(
        f"width {width:2d}: capture {fraction.min():.4f} to {fraction.max():.4f}"
        f"  ->  {100 * (fraction.max() / fraction.min() - 1):.1f}% variation"
    )

ax.legend(frameon=False, ncols=4, fontsize=8)
ax.set(xlabel="Dispersion [pix]", ylabel="Fraction of flux\ninside aperture");

A width-5 aperture captures between 80 and 85 percent of the light depending on wavelength, a
**6 percent** error that varies smoothly over hundreds of pixels, the shape of a real spectral
feature. Widening to 11 pixels cuts it to 1.4 percent, and to 15 pixels to 1.0 percent. The
equivalent width of a narrow line barely notices such a loss, but anything that compares the
continuum across wavelength does, which argues for a **wide** boxcar aperture even at some cost in
signal-to-noise, or for an extraction without a hard aperture edge.

## Optimal extraction

`HorneExtract` implements the algorithm of
[Horne (1986)](https://ui.adsabs.harvard.edu/abs/1986PASP...98..609H/abstract). It weights each pixel by the
**spatial profile**, the expected fraction of the source's light in that row, divided by the
pixel's **variance**, so the pixels near the peak dominate and the noisy wings contribute little
without being discarded. Because the weights depend on the variance, it cannot run without an
uncertainty, which the background-subtracted frame carries. The extracted spectrum is read from its
`spectrum` property, as for the boxcar.

In [ ]:
horne = HorneExtract(image, trace)
horne_spectrum = horne.spectrum

horne_flux = horne_spectrum.flux.value
horne_error = horne_spectrum.uncertainty.array

clean = slice(300, 700)
print(f"{'method':>22}  {'median flux':>12}  {'median sigma':>13}  {'median SNR':>11}")
for label, f, e in (
    ("boxcar, width 11", flux, error),
    ("Horne", horne_flux, horne_error),
):
    print(
        f"{label:>22}  {np.median(f[clean]):>12.0f}  {np.median(e[clean]):>13.1f}  "
        f"{np.median(f[clean] / e[clean]):>11.1f}"
    )

The two are level, a median signal-to-noise of 491 for the boxcar against 485 for Horne, and that
is the expected answer. Optimal extraction pays off where read noise and sky dominate, on faint
sources. At about 240 000 e⁻ per column this source is photon-limited, and when its own shot noise
dominates there is little for the weighting to recover. The fluxes differ, though:

In [ ]:
fig, axs = plt.subplots(
    2, figsize=(8, 3.8), sharex="all", constrained_layout=True, height_ratios=[2, 1]
)

axs[0].plot(flux, lw=0.7, label="boxcar, width 11")
axs[0].plot(horne_flux, lw=0.7, label="Horne")
axs[0].legend(frameon=False, fontsize=8)
axs[0].set_ylabel("Flux [e$^-$]")

axs[1].axhline(1, c="0.6", lw=0.8, ls="--")
axs[1].plot(horne_flux / flux, lw=0.7, c="C2")
axs[1].set(
    xlim=(0, nx - 1),
    ylim=(0.7, 1.1),
    xlabel="Dispersion [pix]",
    ylabel="Horne / boxcar",
)

print(f"flux ratio: median {np.median((horne_flux / flux)[clean]):.3f}")

Horne returns about 3 percent less flux than the boxcar, and the ratio drifts slightly across the
spectrum. The two methods weight the profile differently and so do not measure quite the same
quantity: the width-11 boxcar captures about 95 percent of the light, and Horne's Gaussian kernel
captures a different fraction that varies with the profile width. Neither is "the" flux; whichever
you choose, apply it consistently.

## Spatial profile options

`spatial_profile` sets how `HorneExtract` models the cross-dispersion profile. The default,
`"gaussian"`, fits one Gaussian to the average profile, optionally with a polynomial background
term (`bkgrd_prof`); it is stable and cheap but assumes the same shape at every wavelength.
`"interpolated_profile"` measures the profile empirically in `n_bins_interpolated_profile`
dispersion bins (default 10) and interpolates between them, so it follows a profile that changes
shape and makes no Gaussian assumption, at the cost of more parameters and no background model.
Since the profile width varies here, the empirical option should describe it better.

In [ ]:
profiles = (
    ("gaussian", "gaussian"),
    ("interpolated, 10 bins", "interpolated_profile"),
    (
        "interpolated, 25 bins",
        {"name": "interpolated_profile", "n_bins_interpolated_profile": 25},
    ),
)

print(
    f"{'spatial_profile':>24}  {'median flux':>12}  {'median sigma':>13}  {'median SNR':>11}"
)
for label, profile in profiles:
    s = HorneExtract(image, trace, spatial_profile=profile).spectrum
    f = s.flux.value
    e = s.uncertainty.array
    print(
        f"{label:>24}  {np.median(f[clean]):>12.0f}  {np.median(e[clean]):>13.1f}  "
        f"{np.median(f[clean] / e[clean]):>11.1f}"
    )

The empirical profiles recover more flux than the Gaussian, at a median signal-to-noise about 3
percent higher, 499 against 485, because the target's wings extend beyond a Gaussian core and the
empirical profile follows them. The 10- and 25-bin profiles agree to a fraction of a percent, so
10 bins already resolve how the profile changes along the dispersion axis. The price is noise:
each bin's profile is measured rather than fitted, and on a faint source that noise goes straight
into the weights. Use it where the profile departs from the model and there is signal to measure
it.

## Summary

- Extraction collapses the background-subtracted frame to a 1D `Spectrum` with an uncertainty, on
  a **pixel** axis; wavelength calibration comes later.
- `BoxcarExtract` reaches its SNR plateau at **width 11** here, wider than a Gaussian's 1.4 FWHM
  because the source is photon-limited and its profile has extended wings, and anything from 11 to 35 is within 1 percent of the
  maximum.
- A fixed aperture's **capture fraction varies with wavelength**, by 6.0 percent across the
  spectrum at width 5 and 1.4 percent at width 11. For work that compares the continuum across
  wavelength, prefer a wide aperture over the nominal optimum.
- `HorneExtract` weights pixels by profile over variance, so it needs the uncertainty that
  rectification and background subtraction propagate.
- **On a bright, photon-limited source Horne buys nothing over a boxcar**: 485 against 491 in
  median SNR here. Optimal extraction pays off in the background-limited regime.
- Boxcar and Horne measure slightly different quantities, and Horne returns about 3 percent less
  flux here. Pick one and apply it consistently.
- `spatial_profile="interpolated_profile"` follows the measured profile instead of a Gaussian and
  gains a little SNR on this source, at the cost of measurement noise in the weights.

Next: [Line Identification](09-line-identification.ipynb), which starts the work of putting a
wavelength on the horizontal axis.